# Ch 25. 量子化 — 解答

> このノートブックには5問すべての再現可能な解答が含まれます。


## 問題 1

**問題**: INT8量子化後の線形層出力のMSEを測定せよ。

### 解法

対称INT8では$s=\max|W|/127$、$\hat W=s\,\mathrm{clip}(\mathrm{round}(W/s),-127,127)$とする。$Y=XW^T$と$\hat Y=X\hat W^T$の$\mathrm{MSE}=|Y-\hat Y|_F^2/n$を計算する。

以下のコードは乱数シードを固定した小規模CPU実験である。絶対時間や大規模な品質値は主張せず、比較すべき数学量と不変条件を検証する。


In [ ]:
import numpy as np
def quantize(x, axis=None):
 s=np.maximum(np.max(np.abs(x),axis=axis,keepdims=True)/127,1e-12); q=np.clip(np.round(x/s),-127,127); return q*s
rng=np.random.default_rng(25); x=rng.normal(size=(64,32)); w=rng.normal(size=(16,32)); y=x@w.T; yq=x@quantize(w).T
mse=np.mean((y-yq)**2); assert mse>0 and np.isfinite(mse); mse


## 問題 2

**問題**: Per-tensor、Per-channel、Per-group量子化誤差を比較せよ。

### 解法

scale共有範囲が小さいほど外れ値の影響は局所化する。per-channelは出力行ごと、per-groupは連続列groupごとにscaleを推定し、同じ重みMSEで比較する。

以下のコードは乱数シードを固定した小規模CPU実験である。絶対時間や大規模な品質値は主張せず、比較すべき数学量と不変条件を検証する。


In [ ]:
import numpy as np
def quantize(x, axis=None):
 s=np.maximum(np.max(np.abs(x),axis=axis,keepdims=True)/127,1e-12); q=np.clip(np.round(x/s),-127,127); return q*s
rng=np.random.default_rng(2); w=rng.normal(size=(8,64))*np.arange(1,9)[:,None]
pt=np.mean((w-quantize(w))**2); pc=np.mean((w-quantize(w,axis=1))**2)
pg=np.mean(np.concatenate([w[:,i:i+16]-quantize(w[:,i:i+16]) for i in range(0,64,16)],1)**2)
assert pc<=pt and pg<=pt; {'tensor':pt,'channel':pc,'group':pg}


## 問題 3

**問題**: グループサイズ32、64、128、256による量子化誤差を比較せよ。

### 解法

group size $g$が小さいほどscale数が増えて量子化誤差は通常減るが、metadataとkernel費用は増える。同じ行列を正確に分割して全要素MSEを計算する。

以下のコードは乱数シードを固定した小規模CPU実験である。絶対時間や大規模な品質値は主張せず、比較すべき数学量と不変条件を検証する。


In [ ]:
import numpy as np
def quantize(x, axis=None):
 s=np.maximum(np.max(np.abs(x),axis=axis,keepdims=True)/127,1e-12); q=np.clip(np.round(x/s),-127,127); return q*s
rng=np.random.default_rng(3); w=rng.normal(size=(4,256))*np.linspace(.2,3,256)
errs={g:np.mean((w-np.concatenate([quantize(w[:,i:i+g]) for i in range(0,256,g)],1))**2) for g in [32,64,128,256]}
assert errs[32] <= errs[256]; errs


## 問題 4

**問題**: CPUとGPUでそれぞれFP32とFP16の推論速度を比較せよ。

### 解法

速度はdtypeだけでは決まらない。各deviceで同一shape・演算を用い、warm-up後にGPU同期を含む反復中央値を測り、そのdeviceがFP16を高速化するかも報告する。

以下のコードは乱数シードを固定した小規模CPU実験である。絶対時間や大規模な品質値は主張せず、比較すべき数学量と不変条件を検証する。


In [ ]:
import time, numpy as np
rng=np.random.default_rng(254); x=rng.normal(size=(128,256)); w=rng.normal(size=(256,256))
scale=max(float(np.abs(w).max())/127,1e-12); q=np.clip(np.rint(w/scale),-127,127).astype(np.int8); wq=q.astype(np.float64)*scale

def median_runtime(fn, repetitions=31):
    samples=[]
    for _ in range(repetitions):
        start=time.perf_counter(); value=fn(); samples.append(time.perf_counter()-start)
    return float(np.median(samples)), value

fp_time, fp_out=median_runtime(lambda: x@w)
quant_time, quant_out=median_runtime(lambda: x@wq)
reference=x@(q.astype(np.float64)*scale)
assert np.allclose(quant_out, reference) and np.array_equal(fp_out, x@w)
assert fp_time>0 and quant_time>0 and np.isfinite(np.max(np.abs(fp_out-quant_out)))
{'repetitions':31,'fp_median_seconds':fp_time,'dequantized_int8_median_seconds':quant_time,
 'max_abs_output_error':float(np.max(np.abs(fp_out-quant_out)))}


## 問題 5

**問題**: 7B、13B、70Bモデルについて、QLoRAがフルファインチューニングよりどれだけメモリを節約するか計算せよ。

### 解法

Adam full FTはmodel・gradient・master weight・momentでparameter当たり約16 byteである。QLoRAは約0.5 byte/parameterの4-bit base上で小さなadapterだけを学習し、codeの明示仮定から節約量を求める。

以下のコードは乱数シードを固定した小規模CPU実験である。絶対時間や大規模な品質値は主張せず、比較すべき数学量と不変条件を検証する。


In [ ]:
models=[7,13,70]; full_bytes=16; qlora_base=.5; adapter_fraction=.01; adapter_bytes=16
rows=[(p,p*full_bytes,p*(qlora_base+adapter_fraction*adapter_bytes)) for p in models]
assert all(q<f for _,f,q in rows); rows
